# Consulta de ocupación del calendario de Airbnb en Málaga.

En este notebook realizamos una consulta de la ocupación de los alojamientos de Airbnb en Málaga, utilizando una ventana deslizante de 30 días con un desplazamiento de 7 días.

Nuestro objetivo con esta consulta es poder analizar la evolución temporal de la ocupación de los airbnb para ello, vamos a calcular:

- El porcentaje medio de la ocupación, indicándonos la fracción de los alojamientos que estuvieron reservados de media.
- El total de las reservas.

# Librerias

In [6]:
import sys
import pathlib
notebook_dir = pathlib.Path.cwd()
project_root = notebook_dir.parent.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f'Directorio raíz añadido al path: {project_root}')

In [7]:
from src.kafka.consumer_kafka import create_kafka_stream_df, consumer_kafka_avro
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, window, avg, sum as spark_sum

# Verificación cola de Apache Kafka.

Antes de iniciar el stream, vamos a comprobar que los datos de calendar están disponibles en la cola de kafka. Para ello, utilizamos el consumidor para leer y mostrar por pantalla los mensajes del topic de calendar, utilizamso un timeout de 10 segundos.

In [ ]:
consumer_kafka_avro(
    topic = 'airbnb_calendar_gold',
    idle_timeout_seconds = 10,
    log_to_file = False
)

2026-05-04 17:35:15,511 [INFO] Escritura de logs en archivo desactivada.
2026-05-04 17:35:15,512 [INFO] --- INICIANDO MONITOR PARA: aribnb_calendar_gold ---
2026-05-04 17:35:15,587 [ERROR] Error en consumer aribnb_calendar_gold: KafkaError{code=UNKNOWN_TOPIC_OR_PART,val=3,str="Subscribed topic not available: aribnb_calendar_gold: Broker: Unknown topic or partition"}
2026-05-04 17:35:15,588 [INFO] ============================================================
2026-05-04 17:35:15,588 [INFO] RESUMEN FINAL - TÓPICO: aribnb_calendar_gold
2026-05-04 17:35:15,588 [INFO] - Total mensajes: 0
2026-05-04 17:35:15,589 [INFO] - Columnas: 0
2026-05-04 17:35:15,589 [INFO] ============================================================


# Sesión de Spark

Creamos la sesión de Spark para el procesamiento del stream. Hemos incluido los paquetes de integración con Apache Kafka y Avro, ya que los datos están serializados en este formato.

In [ ]:
spark = (SparkSession.builder
        .appName("Notebook_Calendar_SlidingWindow")
        .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1,org.apache.spark:spark-avro_2.13:4.1.1")
        .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark Session iniciada con éxito.")

Spark Session iniaca con éxito.


# Consulta con Spark Streaming.

Hemos definido el proceamiento del streaming en tres pasos:

1. Lectura del stream desde el topic de kafka usando la función `create_kafka_stream_df`
2. Transformamos las variables `date` a `timestamp` para poder utilizarla como Event Time, y `booked` a entero.
3. Aplicamos una ventana de 30 días con un desplazamiento de 7 días, hemos introducido un watermark de 7 días para gestionar la llegada de los eventos tardios. Después, hemos calculado la ocupación media y el total de reservar por ventana.

In [ ]:
# Read stream using imported function
df_calendar_raw = create_kafka_stream_df(spark, "airbnb_calendar_gold")

# Apply needed transformations and date cast
df_processed = (df_calendar_raw
        .withColumn("event_timestamp", col("date").cast("timestamp"))
        .withColumn("booked", col("booked").cast("integer"))
)

# Window and calculate availability
df_group = (df_processed
        .withWatermark("event_timestamp", "7 days")
        .groupBy(
            window(col("event_timestamp"), "30 days", "7 days")
        ).agg(
            (avg(col("booked")) * 100).alias("ocupacion_media"),
            spark_sum(col("booked")).alias("reservas_totales")
        )
)

# Sort chronologically
df_final = df_group.orderBy(col("window.start").desc())

# Inicio del stream.

Lanzamos la consulta de Spark en modo `complete`, por lo que en cada micro-bathc se reescribe la tabla completa de resultados. Esto se almacenan en memoria bajo el nombre de `tabla_resultados_calendar` para poder consultarlo posteriormente con SQL.

In [ ]:
# Inicializate stream
query_calendar = (df_final.writeStream
        .outputMode("complete")
        .format("memory")
        .queryName("tabla_resultados_calendar")
        .start()
)

26/05/04 17:35:25 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/f5/smc33r0154nfhx6x9ky9x8f00000gp/T/temporary-b161ed80-234d-4ad7-948f-e2e9f675a4b9. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/04 17:35:25 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/04 17:35:25 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


# Consulta de resultados.

Despues de inciar el stream y procesar los primeros mico-batches, hemos consultado la tabla en memoria con Spark SQL para visualizar la evolución de la ocupación por ventana temporal.

In [ ]:
# Query
spark.sql("SELECT * FROM tabla_resultados_calendar").show(truncate=False)

+------------------------------------------+-------------------+----------------+
|window                                    |ocupacion_media_pct|reservas_totales|
+------------------------------------------+-------------------+----------------+
|{2026-09-24 02:00:00, 2026-10-24 02:00:00}|48.63942076727747  |28349           |
|{2026-09-17 02:00:00, 2026-10-17 02:00:00}|48.80901474477756  |61637           |
|{2026-09-10 02:00:00, 2026-10-10 02:00:00}|48.859892938027585 |94925           |
|{2026-09-03 02:00:00, 2026-10-03 02:00:00}|48.90269103775383  |128261          |
|{2026-08-27 02:00:00, 2026-09-26 02:00:00}|48.82025941939469  |142272          |
|{2026-08-20 02:00:00, 2026-09-19 02:00:00}|48.788346716079886 |142179          |
|{2026-08-13 02:00:00, 2026-09-12 02:00:00}|48.93006657058541  |142592          |
|{2026-08-06 02:00:00, 2026-09-05 02:00:00}|48.85354471209938  |142369          |
|{2026-07-30 02:00:00, 2026-08-29 02:00:00}|48.859035069658916 |142385          |
|{2026-07-23 02:

26/05/04 17:39:41 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/04 17:39:41 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/04 17:39:42 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/04 17:39:43 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/04 17:39:44 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/04 17:39:45 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:909